# Agentic System

### *Nina Haide*

## Introduction

This project implements a ReAct-style agentic AI system called the **AI Career Advisor**
that autonomously orchestrates a multi-step career guidance workflow for
candidates entering or transitioning into the AI job market. Rather than
requiring the user to manually run individual tools, the agent reasons through
a sequence of decisions - extracting skills, matching archetypes, retrieving
semantically similar job postings via RAG, generating a personalised
recommendation and assembling a final career report - maintaining explicit
state across all steps.

The system directly integrates tools built across the full project series: the
skill taxonomy from Project 3, the job market archetypes and DistilBERT
embeddings from Project 4, the GPT-2 recommendation generator from Project 5,
and the Adzuna jobs API pipeline. A FAISS vector index over the 1,088 enriched European job postings enables semantic retrieval of the most
relevant real-world examples to ground each recommendation. This project
represents the synthesis of the full analytical pipeline into a single
autonomous system, demonstrating how individual ML components can be composed
into a reasoning agent capable of end-to-end task completion.

## Agentic Task Definition and System Scope

### Goal
The AI Career Advisor is a single-agent ReAct system that takes a candidate's
resume text or skill description as input and autonomously orchestrates a
multi-step career guidance workflow. The agent reasons about the candidate's
profile, matches it to one of five AI job market archetypes, retrieves
semantically similar real job postings as grounding evidence, generates a
personalised recommendation and assembles a final career report - all without
manual intervention between steps.

### Agent Type
Single-agent ReAct (Reason + Act) architecture. The agent maintains an explicit
reasoning trace, deciding which tool to call next based on the observation from
the previous step. This pattern was introduced by Yao et al. (2022) and is the
foundation of modern tool-using language agents.

### Agent Decision Scope
- Whether the input contains sufficient skill signal to proceed or requires
  a clarification response
- Which archetype the candidate maps to and at what confidence level
- Whether to route to a strong fit, partial fit or no-fit workflow
- Whether to retrieve live job postings or rely on the RAG index alone
- When enough information has been gathered to assemble the final report

### Available Tools

| Tool | Description |
|------|-------------|
| `extract_skills(text)` | Extracts skills from free-text input using the Project 3 taxonomy |
| `match_archetype(skills)` | Computes cosine similarity to 5 archetype centroids, returns best match and fit level |
| `rag_retrieve(archetype, query_vec)` | Queries FAISS index of Project 4 job embeddings, returns top 3 semantically similar postings |
| `generate_recommendation(skills, archetype, fit, context)` | Calls fine-tuned GPT-2 from Project 5, conditioned on retrieved job context |
| `fetch_live_jobs(archetype, country)` | Fetches current Adzuna job postings matching the archetype |
| `get_salary_range(archetype)` | Returns salary statistics from Project 2 DACH dataset |

### Memory and State
The agent maintains a state dictionary across all reasoning steps:

```python
state = {
    "input_text":      str,    # original user input
    "skills":          list,   # extracted skills
    "archetype":       str,    # matched persona name
    "fit_score":       float,  # cosine similarity score
    "fit_level":       str,    # strong / partial / none
    "rag_context":     list,   # retrieved similar job postings
    "recommendation":  str,    # generated recommendation text
    "live_jobs":       list,   # current Adzuna postings
    "salary_range":    dict,   # min / median / max EUR
    "reasoning_trace": list,   # full Thought-Action-Observation log
}
```

### System Boundaries
The AI Career Advisor operates on text input only - it does not access files,
browse the web autonomously or store data between sessions. Tool calls are
executed sequentially in a single session. The Adzuna API call is the only
external dependency at runtime; all other tools operate on local data and
models built in previous projects. The agent will not make hiring
recommendations or rank candidates against each other - it is designed
exclusively as a candidate-facing career orientation tool.

## Agent Architecture Design

### Agent Persona
The AI Career Advisor acts as a knowledgeable career guidance specialist with
deep expertise in the European AI job market. It is direct, evidence-based and
constructive - grounding every recommendation in real job market data rather
than generic career advice. When a strong match is found it confirms and
encourages; when the match is weak it identifies gaps and provides a concrete
development pathway.

### Components

**Reasoning Loop (ReAct)**
The agent follows a Thought → Action → Observation cycle at each step. It
explicitly logs its reasoning before every tool call, enabling full
transparency of the decision process and a traceable audit trail of how
the final recommendation was reached.

**Memory and State**
A single state dictionary persists across all steps within a session,
accumulating extracted skills, archetype match, RAG context, generated
recommendation, live jobs and salary data. There is no cross-session
memory - each run starts fresh. This is appropriate for a candidate-facing
tool where privacy and session isolation are important.

**Tool Registry**
Six tools are registered and callable by the agent. Each tool is a
standalone Python function with a defined input/output contract. The agent
selects tools sequentially based on the observation from the previous step
rather than calling all tools unconditionally.

**Output Assembler**
A final step collects all accumulated state into a structured career report - combining the archetype match, recommendation text, RAG-retrieved job
examples, live postings and salary range into a single readable output.

### Architecture Diagram

### Design Choices and Tradeoffs

**Sequential over parallel tool execution**
Tools are called in a fixed sequence rather than in parallel. This simplifies
state management and makes the reasoning trace easier to follow and debug.
The tradeoff is that the Adzuna API call and GPT-2 generation run
sequentially, adding latency - acceptable for a single-user academic system
but not for production deployment.

**Single-agent over multi-agent**
A single ReAct agent was chosen over a multi-agent architecture because the
tools are well-defined, the workflow is linear and the added coordination
overhead of multiple agents would not improve output quality at this scale.
A multi-agent design would be warranted if tools needed to run in parallel
or if specialised sub-agents required independent reasoning loops.

**Local models over API calls**
Skill extraction, archetype matching, RAG retrieval and recommendation
generation all run locally using models from previous projects. This avoids
API costs, ensures reproducibility and keeps the system self-contained.
The tradeoff is higher local compute requirements and lower generation
quality compared to a cloud-hosted LLM.

**FAISS over a hosted vector database**
FAISS runs entirely in memory with no external dependencies, making it
ideal for a self-contained academic project. The tradeoff is that the index
is not persistent between sessions and must be rebuilt each run - acceptable
given the 1,088-document corpus size.

**Threshold-based routing**
Fit level is determined by fixed cosine similarity thresholds (≥0.4 strong,
≥0.2 partial, <0.2 none) rather than a learned classifier. This keeps the
routing logic transparent and interpretable, which is important for a
career guidance system where candidates should understand why they received
a particular recommendation.

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

In [ ]:
pip freeze > requirements.txt